In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
admissions = pd.read_csv("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/admissions.csv")

In [3]:
patients = pd.read_csv("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/patients.csv")

In [4]:
print(admissions.head())
admissions.info()

   subject_id   hadm_id            admittime            dischtime deathtime  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00       NaN   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00       NaN   
2    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00       NaN   
3    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00       NaN   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00       NaN   

   admission_type admit_provider_id      admission_location  \
0          URGENT            P49AFC  TRANSFER FROM HOSPITAL   
1        EW EMER.            P784FA          EMERGENCY ROOM   
2        EW EMER.            P19UTS          EMERGENCY ROOM   
3        EW EMER.            P06OTX          EMERGENCY ROOM   
4  EU OBSERVATION            P39NWO          EMERGENCY ROOM   

  discharge_location insurance language marital_status   race  \
0               HOME  Medicaid  English        WIDOWED  WHITE   
1               

# Preprocessing

In [5]:
admissions['hadm_id'] = pd.to_datetime(admissions['hadm_id'])
admissions['admittime'] = pd.to_datetime(admissions['admittime'])        
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

# Calculating & labeling 30 days readmissions

In [6]:
df = admissions.sort_values(['subject_id', 'admittime'])

# Last admission for subjects
df['is_last'] = df.groupby('subject_id')['admittime'].transform('max') == df['admittime']

# Labeling 0 or 1 for 30 days readmission
df['next_admit_time'] = df.groupby('subject_id')['admittime'].shift(-1)
df['days_for_admission']= (df['next_admit_time']-df['dischtime']).dt.days
df['readmission_30day'] = ((df['days_for_admission']<=30)&(df['days_for_admission']>=0)).astype(int)


## Verifying labeling accuracy

In [7]:
df[['subject_id',	'hadm_id',	'admittime',	'dischtime','next_admit_time','is_last','days_for_admission','readmission_30day']]

,subject_id,hadm_id,admittime,dischtime,next_admit_time,is_last,days_for_admission,readmission_30day
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,2180-06-26 18:27:00,False,50.0,0
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,2180-07-23 12:35:00,False,25.0,1
3,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,2180-08-05 23:44:00,False,11.0,1
2,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaT,True,NaN,0
4,10000068,1970-01-01 00:00:00.025022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaT,True,NaN,0
...,...,...,...,...,...,...,...,...
546024,19999828,1970-01-01 00:00:00.029734428,2147-07-18 16:23:00,2147-08-04 18:10:00,2149-01-08 16:44:00,False,522.0,0
546023,19999828,1970-01-01 00:00:00.025744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaT,True,NaN,0
546026,19999840,1970-01-01 00:00:00.026071774,2164-07-25 00:27:00,2164-07-28 12:15:00,2164-09-10 13:47:00,False,44.0,0
546025,19999840,1970-01-01 00:00:00.021033226,2164-09-10 13:47:00,2164-09-17 13:42:00,NaT,True,NaN,0


In [8]:
admissions[['subject_id',	'hadm_id',	'admittime',	'dischtime']].sort_values(['subject_id', 'admittime'])

,subject_id,hadm_id,admittime,dischtime
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00
3,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00
2,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00
4,10000068,1970-01-01 00:00:00.025022803,2160-03-03 23:16:00,2160-03-04 06:26:00
...,...,...,...,...
546024,19999828,1970-01-01 00:00:00.029734428,2147-07-18 16:23:00,2147-08-04 18:10:00
546023,19999828,1970-01-01 00:00:00.025744818,2149-01-08 16:44:00,2149-01-18 17:00:00
546026,19999840,1970-01-01 00:00:00.026071774,2164-07-25 00:27:00,2164-07-28 12:15:00
546025,19999840,1970-01-01 00:00:00.021033226,2164-09-10 13:47:00,2164-09-17 13:42:00


In [9]:
print(f"Total death counts in admission as per deathtime column is {546028 - (df['deathtime'].isna().sum())} and total alive is {(df['deathtime'].isna().sum())}")

Total death counts in admission as per deathtime column is 11790 and total alive is 534238


In [10]:
print(f"Total death counts as per discharge_location column is {(df['discharge_location'] == 'DIED').sum()}")

Total death counts as per discharge_location column is 11721


In [11]:
dfdeaths = df[df['deathtime'].notna()]

dfdeaths

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,is_last,next_admit_time,days_for_admission,readmission_30day
71,10001843,1970-01-01 00:00:00.026133978,2134-12-05 00:10:00,2134-12-06 12:54:00,2134-12-06 12:54:00,URGENT,P67ATB,TRANSFER FROM HOSPITAL,DIED,Medicare,English,SINGLE,WHITE,NaN,NaN,1,True,NaT,NaN,0
85,10001884,1970-01-01 00:00:00.026184834,2131-01-07 20:39:00,2131-01-20 05:15:00,2131-01-20 05:15:00,OBSERVATION ADMIT,P49AFC,EMERGENCY ROOM,DIED,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,2131-01-07 13:36:00,2131-01-07 22:13:00,1,True,NaT,NaN,0
121,10002155,1970-01-01 00:00:00.020345487,2131-03-09 20:33:00,2131-03-10 01:55:00,2131-03-10 21:53:00,EW EMER.,P579JR,EMERGENCY ROOM,DIED,Medicare,English,MARRIED,WHITE,2131-03-09 19:14:00,2131-03-09 21:33:00,1,True,NaT,NaN,0
230,10003400,1970-01-01 00:00:00.023559586,2137-08-04 00:07:00,2137-09-02 17:05:00,2137-09-02 17:05:00,URGENT,P32CSX,TRANSFER FROM HOSPITAL,DIED,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,NaN,NaN,1,True,NaT,NaN,0
255,10003637,1970-01-01 00:00:00.028317408,2150-05-14 19:51:00,2150-05-22 16:25:00,2150-05-22 16:25:00,EW EMER.,P46834,WALK-IN/SELF REFERRAL,DIED,Medicare,English,DIVORCED,PORTUGUESE,2150-05-14 18:07:00,2150-05-14 21:59:00,1,True,NaT,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545708,19994505,1970-01-01 00:00:00.020026892,2185-11-13 17:22:00,2185-11-17 22:45:00,2185-11-17 00:00:00,EU OBSERVATION,P73FS2,EMERGENCY ROOM,NaN,Medicare,Russian,MARRIED,WHITE,2185-11-13 11:12:00,2185-11-13 22:00:00,1,True,NaT,NaN,0
545764,19995127,1970-01-01 00:00:00.027369164,2138-06-06 18:00:00,2138-06-12 01:48:00,2138-06-12 01:48:00,EW EMER.,P25XX5,EMERGENCY ROOM,DIED,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,2138-06-06 13:56:00,2138-06-06 19:40:00,1,True,NaT,NaN,0
545794,19996061,1970-01-01 00:00:00.026115327,2118-07-25 17:55:00,2118-07-29 13:00:00,2118-07-29 13:00:00,EW EMER.,P416B5,EMERGENCY ROOM,DIED,Medicare,English,SINGLE,WHITE,2118-07-25 14:03:00,2118-07-27 00:28:00,1,True,NaT,NaN,0
545979,19999297,1970-01-01 00:00:00.021439025,2162-08-14 23:55:00,2162-08-23 04:16:00,2162-08-23 04:16:00,OBSERVATION ADMIT,P94E7F,EMERGENCY ROOM,DIED,Other,English,SINGLE,MULTIPLE RACE/ETHNICITY,2162-08-14 18:26:00,2162-08-15 02:01:00,1,True,NaT,NaN,0


#### Death counts on "discharge_location" column is lower than "deathtime" column. Moreover, for hadm_id 1970-01-01 00:00:00.020026892, deathtime is mentioned but discharge_location is not labeled DIED, therefore moving ahead with deathtime column as the accurate indicator of Death.

## Excluding admissions resulting in death

In [12]:
df = df[df['deathtime'].isna()]
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,is_last,next_admit_time,days_for_admission,readmission_30day
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,False,2180-06-26 18:27:00,50.0,0
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0,False,2180-07-23 12:35:00,25.0,1
3,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0,False,2180-08-05 23:44:00,11.0,1
2,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0,True,NaT,NaN,0
4,10000068,1970-01-01 00:00:00.025022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0,True,NaT,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545995,19999784,1970-01-01 00:00:00.021364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,English,SINGLE,BLACK/AFRICAN AMERICAN,NaN,NaN,0,True,NaT,NaN,0
546024,19999828,1970-01-01 00:00:00.029734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,2147-07-17 17:18:00,2147-07-18 17:34:00,0,False,2149-01-08 16:44:00,522.0,0
546023,19999828,1970-01-01 00:00:00.025744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,2149-01-08 09:11:00,2149-01-08 18:12:00,0,True,NaT,NaN,0
546026,19999840,1970-01-01 00:00:00.026071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,English,WIDOWED,WHITE,2164-07-24 21:16:00,2164-07-25 01:20:00,0,False,2164-09-10 13:47:00,44.0,0


## Merging with patients

In [13]:
patients

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13
...,...,...,...,...,...,...
364622,19999828,F,46,2147,2017 - 2019,NaN
364623,19999829,F,28,2186,2008 - 2010,NaN
364624,19999840,M,58,2164,2008 - 2010,2164-09-17
364625,19999914,F,49,2158,2017 - 2019,NaN


In [14]:
df = df.merge(patients, on = 'subject_id', how = 'left')
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,hospital_expire_flag,is_last,next_admit_time,days_for_admission,readmission_30day,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,False,2180-06-26 18:27:00,50.0,0,F,52,2180,2014 - 2016,2180-09-09
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,0,False,2180-07-23 12:35:00,25.0,1,F,52,2180,2014 - 2016,2180-09-09
2,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,0,False,2180-08-05 23:44:00,11.0,1,F,52,2180,2014 - 2016,2180-09-09
3,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,0,True,NaT,NaN,0,F,52,2180,2014 - 2016,2180-09-09
4,10000068,1970-01-01 00:00:00.025022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,0,True,NaT,NaN,0,F,19,2160,2008 - 2010,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534233,19999784,1970-01-01 00:00:00.021364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,...,0,True,NaT,NaN,0,M,57,2119,2017 - 2019,NaN
534234,19999828,1970-01-01 00:00:00.029734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,...,0,False,2149-01-08 16:44:00,522.0,0,F,46,2147,2017 - 2019,NaN
534235,19999828,1970-01-01 00:00:00.025744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,...,0,True,NaT,NaN,0,F,46,2147,2017 - 2019,NaN
534236,19999840,1970-01-01 00:00:00.026071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,...,0,False,2164-09-10 13:47:00,44.0,0,M,58,2164,2008 - 2010,2164-09-17


# Selecting only admissions of patients who were confirmed dead after discharge

In [15]:
death_after_discharge = df[df['dod'].notnull()]
death_after_discharge[['subject_id',	'hadm_id',	'admittime',	'dischtime',	'deathtime', 'discharge_location' , 'dod']]

,subject_id,hadm_id,admittime,dischtime,deathtime,discharge_location,dod
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,HOME,2180-09-09
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,HOME,2180-09-09
2,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,HOME,2180-09-09
3,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,HOSPICE,2180-09-09
5,10000084,1970-01-01 00:00:00.023052089,2160-11-21 01:56:00,2160-11-25 14:52:00,NaN,HOME HEALTH CARE,2161-02-13
...,...,...,...,...,...,...,...
534187,19999204,1970-01-01 00:00:00.029046609,2146-05-30 16:43:00,2146-06-08 20:20:00,NaN,HOME,2146-08-29
534188,19999287,1970-01-01 00:00:00.025875727,2191-12-29 07:15:00,2192-01-11 19:00:00,NaN,HOME HEALTH CARE,2197-09-02
534189,19999287,1970-01-01 00:00:00.022997012,2197-07-26 03:29:00,2197-07-31 14:00:00,NaN,HOME HEALTH CARE,2197-09-02
534190,19999287,1970-01-01 00:00:00.020175828,2197-08-03 20:58:00,2197-08-18 15:37:00,NaN,HOSPICE,2197-09-02


## Identifying patients that died within 30 days of their last discharge date and eliminating them

In [16]:
death_after_discharge.info()


<class 'pandas.core.frame.DataFrame'>
Index: 133176 entries, 0 to 534236
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            133176 non-null  int64         
 1   hadm_id               133176 non-null  datetime64[ns]
 2   admittime             133176 non-null  datetime64[ns]
 3   dischtime             133176 non-null  datetime64[ns]
 4   deathtime             0 non-null       object        
 5   admission_type        133176 non-null  object        
 6   admit_provider_id     133174 non-null  object        
 7   admission_location    133176 non-null  object        
 8   discharge_location    108864 non-null  object        
 9   insurance             132706 non-null  object        
 10  language              133015 non-null  object        
 11  marital_status        131292 non-null  object        
 12  race                  133176 non-null  object        
 13  edre

In [17]:
# Changing date of death column to date time
death_after_discharge['dod'] = pd.to_datetime(death_after_discharge['dod'], errors='coerce')

#Find the date difference for last admissions
death_after_discharge['days_to_death'] = (
     death_after_discharge['dod'] - death_after_discharge['dischtime'].dt.normalize()
).dt.days

# Initializing the column with 'no'
death_after_discharge['death_30_days_disch'] = 'not_last'

# Applying the logic (if its the last admission of a patient and the patient died within 30 days of discharge then "yes" 
# And if patient died 30 days later than last discharge date then ">=30"

death_after_discharge.loc[
    death_after_discharge['is_last'] & death_after_discharge['days_to_death'].notna() & (death_after_discharge['days_to_death'] <= 30), 
'death_30_days_disch'] = 'is_last_30'

death_after_discharge.loc[
    death_after_discharge['is_last'] & death_after_discharge['days_to_death'].notna() & (death_after_discharge['days_to_death'] >= 30),
    'death_30_days_disch'
] = 'is_last_30+'


/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62100/873139178.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  death_after_discharge['dod'] = pd.to_datetime(death_after_discharge['dod'], errors='coerce')
/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62100/873139178.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  death_after_discharge['days_to_death'] = (
/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62100/873139178.py:10: SettingWithCopyWarning: 
A value is tryi

In [18]:
death_after_discharge

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,next_admit_time,days_for_admission,readmission_30day,gender,anchor_age,anchor_year,anchor_year_group,dod,days_to_death,death_30_days_disch
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,2180-06-26 18:27:00,50.0,0,F,52,2180,2014 - 2016,2180-09-09,125,not_last
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,2180-07-23 12:35:00,25.0,1,F,52,2180,2014 - 2016,2180-09-09,74,not_last
2,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,2180-08-05 23:44:00,11.0,1,F,52,2180,2014 - 2016,2180-09-09,46,not_last
3,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,NaT,NaN,0,F,52,2180,2014 - 2016,2180-09-09,33,is_last_30+
5,10000084,1970-01-01 00:00:00.023052089,2160-11-21 01:56:00,2160-11-25 14:52:00,NaN,EW EMER.,P42H7G,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,...,2160-12-28 05:11:00,32.0,0,M,72,2160,2017 - 2019,2161-02-13,80,not_last
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534187,19999204,1970-01-01 00:00:00.029046609,2146-05-30 16:43:00,2146-06-08 20:20:00,NaN,OBSERVATION ADMIT,P23ZZK,TRANSFER FROM HOSPITAL,HOME,Medicare,...,NaT,NaN,0,M,61,2146,2017 - 2019,2146-08-29,82,is_last_30+
534188,19999287,1970-01-01 00:00:00.025875727,2191-12-29 07:15:00,2192-01-11 19:00:00,NaN,SURGICAL SAME DAY ADMISSION,P215WX,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicare,...,2197-07-26 03:29:00,2022.0,0,F,71,2191,2008 - 2010,2197-09-02,2061,not_last
534189,19999287,1970-01-01 00:00:00.022997012,2197-07-26 03:29:00,2197-07-31 14:00:00,NaN,EW EMER.,P5069B,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,...,2197-08-03 20:58:00,3.0,1,F,71,2191,2008 - 2010,2197-09-02,33,not_last
534190,19999287,1970-01-01 00:00:00.020175828,2197-08-03 20:58:00,2197-08-18 15:37:00,NaN,EW EMER.,P25YQO,EMERGENCY ROOM,HOSPICE,Medicare,...,NaT,NaN,0,F,71,2191,2008 - 2010,2197-09-02,15,is_last_30


# Checking for accurate labeling

In [19]:
death_after_discharge[death_after_discharge["death_30_days_disch"]=='is_last_30'][['subject_id',	'hadm_id',	'admittime',	'dischtime', 'deathtime', 'discharge_location', 'dod','days_to_death', 'death_30_days_disch','is_last','readmission_30day']]

,subject_id,hadm_id,admittime,dischtime,deathtime,discharge_location,dod,days_to_death,death_30_days_disch,is_last,readmission_30day
19,10000690,1970-01-01 00:00:00.026146595,2152-01-28 23:40:00,2152-01-30 15:56:00,NaN,SKILLED NURSING FACILITY,2152-01-30,0,is_last_30,True,0
33,10000935,1970-01-01 00:00:00.025849114,2187-10-10 19:09:00,2187-10-26 17:00:00,NaN,HOSPICE,2187-11-12,17,is_last_30,True,0
42,10000980,1970-01-01 00:00:00.020897796,2193-08-15 01:01:00,2193-08-17 15:07:00,NaN,HOME HEALTH CARE,2193-08-26,9,is_last_30,True,0
118,10002131,1970-01-01 00:00:00.024065018,2128-03-17 14:53:00,2128-03-19 16:25:00,NaN,HOSPICE,2128-03-21,2,is_last_30,True,0
158,10002557,1970-01-01 00:00:00.026331577,2160-03-04 23:46:00,2160-03-07 17:06:00,NaN,HOSPICE,2160-03-21,14,is_last_30,True,0
...,...,...,...,...,...,...,...,...,...,...,...
534116,19997886,1970-01-01 00:00:00.020793010,2186-11-12 07:10:00,2186-12-10 20:35:00,NaN,HOSPICE,2186-12-11,1,is_last_30,True,0
534139,19998330,1970-01-01 00:00:00.024096838,2178-11-27 21:51:00,2178-12-01 17:10:00,NaN,HOME HEALTH CARE,2178-12-08,7,is_last_30,True,0
534168,19998843,1970-01-01 00:00:00.024842066,2187-02-05 09:27:00,2187-02-08 17:28:00,NaN,DIED,2187-02-08,0,is_last_30,True,0
534174,19998878,1970-01-01 00:00:00.021643535,2132-12-20 06:00:00,2132-12-26 13:45:00,NaN,HOME HEALTH CARE,2133-01-03,8,is_last_30,True,0


In [20]:
death_after_discharge[(death_after_discharge["is_last"]==True)&(death_after_discharge["readmission_30day"]==1)][['subject_id',	'hadm_id',	'admittime',	'dischtime', 'deathtime', 'discharge_location', 'dod','days_to_death', 'death_30_days_disch','is_last','readmission_30day']]

,subject_id,hadm_id,admittime,dischtime,deathtime,discharge_location,dod,days_to_death,death_30_days_disch,is_last,readmission_30day


In [21]:
death_after_discharge[(death_after_discharge["is_last"]==True)&(death_after_discharge["death_30_days_disch"]== "is_last_30+")][['subject_id',	'hadm_id',	'admittime',	'dischtime', 'deathtime', 'discharge_location', 'dod','days_to_death', 'death_30_days_disch','is_last','readmission_30day']]

,subject_id,hadm_id,admittime,dischtime,deathtime,discharge_location,dod,days_to_death,death_30_days_disch,is_last,readmission_30day
3,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,HOSPICE,2180-09-09,33,is_last_30+,True,0
6,10000084,1970-01-01 00:00:00.029888819,2160-12-28 05:11:00,2160-12-28 16:07:00,NaN,NaN,2161-02-13,47,is_last_30+,True,0
34,10000947,1970-01-01 00:00:00.027880650,2121-05-09 10:02:00,2121-05-13 16:20:00,NaN,HOME,2121-08-23,102,is_last_30+,True,0
68,10001667,1970-01-01 00:00:00.022672901,2173-08-22 17:16:00,2173-08-24 16:45:00,NaN,HOME HEALTH CARE,2174-04-23,242,is_last_30+,True,0
95,10001919,1970-01-01 00:00:00.029897682,2124-04-20 00:00:00,2124-04-21 13:47:00,NaN,HOME,2124-12-20,243,is_last_30+,True,0
...,...,...,...,...,...,...,...,...,...,...,...
534113,19997760,1970-01-01 00:00:00.021257506,2190-12-03 20:51:00,2190-12-06 18:12:00,NaN,SKILLED NURSING FACILITY,2191-10-08,306,is_last_30+,True,0
534154,19998497,1970-01-01 00:00:00.021557581,2145-07-24 23:31:00,2145-08-01 13:04:00,NaN,HOME HEALTH CARE,2146-02-24,207,is_last_30+,True,0
534157,19998562,1970-01-01 00:00:00.026846592,2166-04-06 20:38:00,2166-04-16 16:20:00,NaN,HOME,2167-03-15,333,is_last_30+,True,0
534160,19998591,1970-01-01 00:00:00.024349193,2185-07-03 20:20:00,2185-08-03 14:42:00,NaN,REHAB,2185-12-05,124,is_last_30+,True,0


In [22]:
death_after_discharge['is_last'].value_counts()

is_last
False    108080
True      25096
Name: count, dtype: int64

In [23]:
death_after_discharge["death_30_days_disch"].value_counts()

death_30_days_disch
not_last       108080
is_last_30+     16638
is_last_30       8458
Name: count, dtype: int64

### Based on the data, patients who died after their discharge had total of 133176 admissions, among which 25,096 were their last admissions. Among 25,096 last admissions, 8458 admissions resulted in death within 30 days of discharge, 16,638 resulted in death 30 days after discharge.

# Excluding 8458 last admissions that resulted in death within 30 days of discharge

In [24]:
exclusion_data = death_after_discharge[death_after_discharge["death_30_days_disch"]== "is_last_30"]
exclusion_data

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,next_admit_time,days_for_admission,readmission_30day,gender,anchor_age,anchor_year,anchor_year_group,dod,days_to_death,death_30_days_disch
19,10000690,1970-01-01 00:00:00.026146595,2152-01-28 23:40:00,2152-01-30 15:56:00,NaN,EW EMER.,P61PLH,EMERGENCY ROOM,SKILLED NURSING FACILITY,Medicare,...,NaT,NaN,0,F,86,2150,2008 - 2010,2152-01-30,0,is_last_30
33,10000935,1970-01-01 00:00:00.025849114,2187-10-10 19:09:00,2187-10-26 17:00:00,NaN,EW EMER.,P40LGD,EMERGENCY ROOM,HOSPICE,Medicare,...,NaT,NaN,0,F,52,2182,2008 - 2010,2187-11-12,17,is_last_30
42,10000980,1970-01-01 00:00:00.020897796,2193-08-15 01:01:00,2193-08-17 15:07:00,NaN,OBSERVATION ADMIT,P55EL5,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,...,NaT,NaN,0,F,73,2186,2008 - 2010,2193-08-26,9,is_last_30
118,10002131,1970-01-01 00:00:00.024065018,2128-03-17 14:53:00,2128-03-19 16:25:00,NaN,EW EMER.,P58SXJ,EMERGENCY ROOM,HOSPICE,Medicare,...,NaT,NaN,0,F,87,2123,2011 - 2013,2128-03-21,2,is_last_30
158,10002557,1970-01-01 00:00:00.026331577,2160-03-04 23:46:00,2160-03-07 17:06:00,NaN,EW EMER.,P15WZ8,PHYSICIAN REFERRAL,HOSPICE,Medicare,...,NaT,NaN,0,F,75,2145,2008 - 2010,2160-03-21,14,is_last_30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534116,19997886,1970-01-01 00:00:00.020793010,2186-11-12 07:10:00,2186-12-10 20:35:00,NaN,EW EMER.,P61YAW,CLINIC REFERRAL,HOSPICE,Medicare,...,NaT,NaN,0,M,67,2181,2011 - 2013,2186-12-11,1,is_last_30
534139,19998330,1970-01-01 00:00:00.024096838,2178-11-27 21:51:00,2178-12-01 17:10:00,NaN,EW EMER.,P797L4,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,...,NaT,NaN,0,F,71,2177,2011 - 2013,2178-12-08,7,is_last_30
534168,19998843,1970-01-01 00:00:00.024842066,2187-02-05 09:27:00,2187-02-08 17:28:00,NaN,EW EMER.,P07KMB,EMERGENCY ROOM,DIED,Medicaid,...,NaT,NaN,0,M,45,2187,2011 - 2013,2187-02-08,0,is_last_30
534174,19998878,1970-01-01 00:00:00.021643535,2132-12-20 06:00:00,2132-12-26 13:45:00,NaN,EW EMER.,P030II,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,...,NaT,NaN,0,M,56,2132,2008 - 2010,2133-01-03,8,is_last_30


In [25]:
df = df[~df["hadm_id"].isin(exclusion_data["hadm_id"])]

In [26]:
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,hospital_expire_flag,is_last,next_admit_time,days_for_admission,readmission_30day,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,1970-01-01 00:00:00.022595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,False,2180-06-26 18:27:00,50.0,0,F,52,2180,2014 - 2016,2180-09-09
1,10000032,1970-01-01 00:00:00.022841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,0,False,2180-07-23 12:35:00,25.0,1,F,52,2180,2014 - 2016,2180-09-09
2,10000032,1970-01-01 00:00:00.029079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,0,False,2180-08-05 23:44:00,11.0,1,F,52,2180,2014 - 2016,2180-09-09
3,10000032,1970-01-01 00:00:00.025742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,0,True,NaT,NaN,0,F,52,2180,2014 - 2016,2180-09-09
4,10000068,1970-01-01 00:00:00.025022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,0,True,NaT,NaN,0,F,19,2160,2008 - 2010,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534233,19999784,1970-01-01 00:00:00.021364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,...,0,True,NaT,NaN,0,M,57,2119,2017 - 2019,NaN
534234,19999828,1970-01-01 00:00:00.029734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,...,0,False,2149-01-08 16:44:00,522.0,0,F,46,2147,2017 - 2019,NaN
534235,19999828,1970-01-01 00:00:00.025744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,...,0,True,NaT,NaN,0,F,46,2147,2017 - 2019,NaN
534236,19999840,1970-01-01 00:00:00.026071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,...,0,False,2164-09-10 13:47:00,44.0,0,M,58,2164,2008 - 2010,2164-09-17


# Merging with features extracted from diagnoses

In [27]:
diagnoses = pd.read_csv("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/diagnoses_icd.csv")
diagnoses

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
6364483,19999987,23865745,7,41401,9
6364484,19999987,23865745,8,78039,9
6364485,19999987,23865745,9,0413,9
6364486,19999987,23865745,10,36846,9


In [28]:
print(f'total unique patients: {diagnoses["subject_id"].nunique()}')
print(f'total unique admissions: {diagnoses['hadm_id'].nunique()}')

total unique patients: 223291
total unique admissions: 545497


In [29]:
# Get indices of min and max seq_num for each hadm_id
first_idx = diagnoses.groupby('hadm_id')['seq_num'].idxmin()
last_idx = diagnoses.groupby('hadm_id')['seq_num'].idxmax()

# Get first and last diagnoses values
result_1 = pd.DataFrame({
    'hadm_id': first_idx.index,
    'diagnoses_first_icd_code': diagnoses.loc[first_idx, 'icd_code'].values,
    'diagnoses_last_icd_code': diagnoses.loc[last_idx, 'icd_code'].values
})

In [30]:
# Get no of unique diagnoses counts, and the most frequent icd code
result_2 = diagnoses.groupby('hadm_id')['icd_code'].agg([
    ('diagnoses_total_count', 'count'),
    ('diagnoses_most_frequent_icd', lambda x: x.value_counts().index[0])
])


In [31]:
# Get last 5 ICD codes as a single concatenated string
last_5_diagnoses = (diagnoses.sort_values(['hadm_id', 'seq_num'])
                   .groupby('hadm_id')
                   .tail(5)
                   .sort_values(['hadm_id', 'seq_num'], ascending=[True, False])  # Reverse seq_num order
                   .groupby('hadm_id')['icd_code']
                   .apply(lambda x: ' '.join(x.astype(str)))
                   .reset_index()
                   .rename(columns={'icd_code': 'diagnoses_last_5_icd_codes'}))

last_5_diagnoses

,hadm_id,diagnoses_last_5_icd_codes
0,20000019,2859 V1649 2724 49390 4019
1,20000024,Y92099 T474X5A H548 Z9181 R270
2,20000034,Z87891 F09 D509 G4700 R339
3,20000041,27800 V1251 V1042 53081 V4586
4,20000045,D6481 G893 Z8616 Z87891 E8339
...,...,...
545492,29999803,M549 R0781 R42 Z8673 Z85828
545493,29999809,V1582 V4582 60000 2724 4019
545494,29999828,79902 25000 2724 4019 V8542
545495,29999928,K5900 D500 I493 I959 Z8241


In [32]:
# Get last 5 ICD codes as a single concatenated string
def pad_to_5_with_last(group):
    # Get all codes, reverse order (most recent first)
    codes = group['icd_code'].tolist()[::-1]
    # Pad with last code if needed
    while len(codes) < 5:
        codes.append(codes[0])
    return ' '.join(str(code) for code in codes[:5])

last_5_diagnoses_nempty = (diagnoses.sort_values(['hadm_id', 'seq_num'])
                    .groupby('hadm_id')
                    .apply(pad_to_5_with_last)
                    .reset_index()
                    .rename(columns={0: 'diagnoses_last_5_icd_codes'}))

last_5_diagnoses_nempty

/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62100/2163355644.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pad_to_5_with_last)


,hadm_id,diagnoses_last_5_icd_codes
0,20000019,2859 V1649 2724 49390 4019
1,20000024,Y92099 T474X5A H548 Z9181 R270
2,20000034,Z87891 F09 D509 G4700 R339
3,20000041,27800 V1251 V1042 53081 V4586
4,20000045,D6481 G893 Z8616 Z87891 E8339
...,...,...
545492,29999803,M549 R0781 R42 Z8673 Z85828
545493,29999809,V1582 V4582 60000 2724 4019
545494,29999828,79902 25000 2724 4019 V8542
545495,29999928,K5900 D500 I493 I959 Z8241


In [ ]:
#Merge extracted diagnoses features in a single df 
diagnoses_admission = result_1.merge(result_2, on = "hadm_id", how ="left").merge(last_5_diagnoses,on = "hadm_id", how ="left").merge(last_5_diagnoses_nempty,on = "hadm_id", how ="left")

In [35]:
diagnoses_admission 

,hadm_id,diagnoses_first_icd_code,diagnoses_last_icd_code,diagnoses_total_count,diagnoses_most_frequent_icd,diagnoses_last_5_icd_codes_x,diagnoses_last_5_icd_codes_y
0,20000019,0389,2859,12,0389,2859 V1649 2724 49390 4019,2859 V1649 2724 49390 4019
1,20000024,D500,Y92099,10,D500,Y92099 T474X5A H548 Z9181 R270,Y92099 T474X5A H548 Z9181 R270
2,20000034,K831,Z87891,28,K831,Z87891 F09 D509 G4700 R339,Z87891 F09 D509 G4700 R339
3,20000041,71536,27800,10,71536,27800 V1251 V1042 53081 V4586,27800 V1251 V1042 53081 V4586
4,20000045,A419,D6481,16,A419,D6481 G893 Z8616 Z87891 E8339,D6481 G893 Z8616 Z87891 E8339
...,...,...,...,...,...,...,...
545492,29999803,I110,M549,30,I110,M549 R0781 R42 Z8673 Z85828,M549 R0781 R42 Z8673 Z85828
545493,29999809,41401,V1582,15,41401,V1582 V4582 60000 2724 4019,V1582 V4582 60000 2724 4019
545494,29999828,27801,79902,8,27801,79902 25000 2724 4019 V8542,79902 25000 2724 4019 V8542
545495,29999928,I472,K5900,11,I472,K5900 D500 I493 I959 Z8241,K5900 D500 I493 I959 Z8241


In [36]:
diagnoses_admission[diagnoses_admission["hadm_id"]==22595853]

,hadm_id,diagnoses_first_icd_code,diagnoses_last_icd_code,diagnoses_total_count,diagnoses_most_frequent_icd,diagnoses_last_5_icd_codes_x,diagnoses_last_5_icd_codes_y
141818,22595853,5723,V1582,8,5723,V1582 30981 29680 496 07070,V1582 30981 29680 496 07070


In [37]:
diagnoses[diagnoses["hadm_id"]==22595853]

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
5,10000032,22595853,6,29680,9
6,10000032,22595853,7,30981,9
7,10000032,22595853,8,V1582,9


In [38]:
#Changing datatype for merging
df['hadm_id'] = df['hadm_id'].astype('int64')

/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62100/1641029900.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['hadm_id'] = df['hadm_id'].astype('int64')


# Merging features from diagnoses with original df

In [39]:
df = df.merge(diagnoses_admission, on = "hadm_id", how = "left")
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,anchor_age,anchor_year,anchor_year_group,dod,diagnoses_first_icd_code,diagnoses_last_icd_code,diagnoses_total_count,diagnoses_most_frequent_icd,diagnoses_last_5_icd_codes_x,diagnoses_last_5_icd_codes_y
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,52,2180,2014 - 2016,2180-09-09,5723,V1582,8.0,5723,V1582 30981 29680 496 07070,V1582 30981 29680 496 07070
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,52,2180,2014 - 2016,2180-09-09,07071,3051,8.0,07071,3051 V08 5715 496 2761,3051 V08 5715 496 2761
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,52,2180,2014 - 2016,2180-09-09,45829,5715,13.0,45829,5715 29680 496 V462 V4986,5715 29680 496 V462 V4986
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,52,2180,2014 - 2016,2180-09-09,07054,78791,10.0,07054,78791 3051 V08 496 2761,78791 3051 V08 496 2761
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,19,2160,2008 - 2010,NaN,30500,30500,1.0,30500,30500,30500 30500 30500 30500 30500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525775,19999784,21364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,...,57,2119,2017 - 2019,NaN,Z5111,Z8619,12.0,Z5111,Z8619 E876 D708 T451X5A D701,Z8619 E876 D708 T451X5A D701
525776,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,...,46,2147,2017 - 2019,NaN,T8131XA,I9581,22.0,T8131XA,I9581 Z1611 B9620 Z87891 Z9049,I9581 Z1611 B9620 Z87891 Z9049
525777,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,...,46,2147,2017 - 2019,NaN,T8141XA,R197,19.0,T8141XA,R197 E60 B954 E876 F419,R197 E60 B954 E876 F419
525778,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,...,58,2164,2008 - 2010,2164-09-17,43491,3051,7.0,43491,3051 2724 4019 43811 34590,3051 2724 4019 43811 34590


In [40]:
df.to_csv('adm_pat_diag.csv', index=False)